# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Print the dataset name and description (accessed as attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each RecordSet, Field, and Column has a unique `@id`. We will enumerate RecordSets, then inspect their fields and columns, printing out their `@id`s and summary metadata.

In [ ]:
# Helper to pretty-print nested dicts
def pretty(obj):
    print(json.dumps(obj, indent=2) if isinstance(obj, dict) else obj)

print("Listing all RecordSets and their fields (showing '@id' wherever possible):\n")

# Get all record sets from the metadata
record_set_metadatas = dataset.metadata.record_sets
for rset in record_set_metadatas:
    print(f"RecordSet @id: {rset['@id']}")
    print(f"  Name: {rset.get('name','<no name>')}")
    print(f"  Description: {rset.get('description','<no description>')}")
    # List fields
    fields = rset.get('fields', [])
    print(f"  Fields:")
    for f in fields:
        # f can be either a field dict or @id (if compact)
        if isinstance(f, dict):
            _id = f.get('@id', '<no @id>')
            name = f.get('name', '<no name>')
        else:
            _id = f
            name = '<name in definition>'
        print(f"    - @id: {_id}, name: {name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** All lookups use the `@id` of the RecordSet.

In [ ]:
# For illustration, extract ALL record sets
# In this dataset, assume record sets have IDs such as 'https://api.app.sen.science/frontiers/7154140/4d8fc139-a513-460d-bfdd-f9b6a00705d5' etc.

record_sets = [rset['@id'] for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Select the first record set as an example:
main_record_set_id = record_sets[0]

print(f"Columns available in RecordSet {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a specific numeric field, normalizing that field, and grouping data, using only `@id`s for field selection.

Replace the field `@id`s below with those relevant from the overview.

In [ ]:
# Replace with a real numeric field's @id from the selected record set, e.g. '@id': 'https://api.app.sen.science/frontiers/7154140/field/peptide_count'
numeric_field_id = None
group_field_id = None

# Inspect columns to pick an example numeric field
for col in dataframes[main_record_set_id].columns:
    # Example: find a field containing 'peptide' or 'MW' (molecular weight) in the ID
    if 'peptide' in col or 'MW' in col or 'abundance' in col:
        numeric_field_id = col
        break
# For group field, look for e.g. sample type or protein class
for col in dataframes[main_record_set_id].columns:
    if 'description' in col or 'accession' in col:
        group_field_id = col
        break
if numeric_field_id is None or group_field_id is None:
    print("Could not automatically determine suitable numeric or group field. Please review columns above and set manually.")

# For demo, continue only if field was found
if numeric_field_id is not None:
    # Remove non-numeric values (coerce)
    df = dataframes[main_record_set_id].copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Set to mean as arbitrary threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Group by another field, e.g., description or accession
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for EDA demo.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use matplotlib and seaborn if available for a histogram and a boxplot on the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if a numeric field was found
if numeric_field_id is not None:
    plt.figure(figsize=(12,5))
    sns.histplot(data=df, x=numeric_field_id, kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.show()
    
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[numeric_field_id])
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.show()
else:
    print("No numeric field selected for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration: 

- We loaded the Croissant dataset using its schema URL and explored its metadata programmatically using `mlcroissant`.
- RecordSets and fields (referenced by their `@id`s) were listed, and one main RecordSet was extracted into a DataFrame for analysis.
- Simple EDA, such as filtering, normalization, and grouping, demonstrated the use of field `@id`s for robust referencing.
- Visualizations showed the distribution of a selected numeric attribute for biological analysis.

This structured workflow ensures reproducibility and compliance with FAIR data principles by referencing data entities via persistent `@id`s.